In [11]:
import pandas as pd
import numpy as np

fy2019 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/H-1B_Disclosure_Data_FY2019.xlsx"

df = pd.read_excel(fy2019)

print(df.shape)
print(df.head())

(664616, 260)
          CASE_NUMBER CASE_STATUS CASE_SUBMITTED DECISION_DATE  \
0  I-200-16092-327771   WITHDRAWN     2016-04-08    2019-04-30   
1  I-203-17188-450729   WITHDRAWN     2017-07-14    2019-05-13   
2  I-203-17229-572307   WITHDRAWN     2017-08-23    2019-04-30   
3  I-203-17356-299648   WITHDRAWN     2017-12-22    2019-08-20   
4  I-203-18008-577576   WITHDRAWN     2018-01-10    2019-04-15   

  ORIGINAL_CERT_DATE      VISA_CLASS                    JOB_TITLE SOC_CODE  \
0                NaT            H-1B  ASSOCIATE CREATIVE DIRECTOR  11-2011   
1                NaT  E-3 Australian  ACCOUNT SUPERVISOR (MOTHER)  11-2011   
2                NaT  E-3 Australian  EXECUTIVE CREATIVE DIRECTOR  11-2011   
3                NaT  E-3 Australian      PROJECT MANAGEMENT LEAD  11-2011   
4                NaT  E-3 Australian        CREATIVE DIRECTOR, UX  11-2011   

                             SOC_TITLE FULL_TIME_POSITION  ...  \
0  ADVERTISING AND PROMOTIONS MANAGERS                

In [13]:
print(df.columns.tolist())

['CASE_NUMBER', 'CASE_STATUS', 'CASE_SUBMITTED', 'DECISION_DATE', 'ORIGINAL_CERT_DATE', 'VISA_CLASS', 'JOB_TITLE', 'SOC_CODE', 'SOC_TITLE', 'FULL_TIME_POSITION', 'PERIOD_OF_EMPLOYMENT_START_DATE', 'PERIOD_OF_EMPLOYMENT_END_DATE', 'TOTAL_WORKER_POSITIONS', 'NEW_EMPLOYMENT', 'CONTINUED_EMPLOYMENT', 'CHANGE_PREVIOUS_EMPLOYMENT', 'NEW_CONCURRENT_EMPLOYMENT', 'CHANGE_EMPLOYER', 'AMENDED_PETITION', 'EMPLOYER_NAME', 'EMPLOYER_BUSINESS_DBA', 'EMPLOYER_ADDRESS1', 'EMPLOYER_ADDRESS2', 'EMPLOYER_CITY', 'EMPLOYER_STATE', 'EMPLOYER_POSTAL_CODE', 'EMPLOYER_COUNTRY', 'EMPLOYER_PROVINCE', 'EMPLOYER_PHONE', 'EMPLOYER_PHONE_EXT', 'NAICS_CODE', 'AGENT_REPRESENTING_EMPLOYER', 'AGENT_ATTORNEY_LAW_FIRM_BUSINESS_NAME', 'AGENT_ATTORNEY_ADDRESS1', 'AGENT_ATTORNEY_ADDRESS2', 'AGENT_ATTORNEY_CITY', 'AGENT_ATTORNEY_STATE', 'AGENT_ATTORNEY_POSTAL_CODE', 'AGENT_ATTORNEY_COUNTRY', 'AGENT_ATTORNEY_PROVINCE', 'AGENT_ATTORNEY_PHONE', 'AGENT_ATTORNEY_PHONE_EXT', 'STATE_OF_HIGHEST_COURT', 'NAME_OF_HIGHEST_STATE_COURT', '

In [15]:
import pandas as pd

df = pd.read_excel(fy2019)

df["EMPLOYER_NAME"] = (
    df["EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

EMPLOYER_NAME
GOOGLE LLC                                                 10009
AMAZON.COM SERVICES, INC.                                   7267
MICROSOFT CORPORATION                                       6040
FACEBOOK, INC.                                              2680
APPLE INC.                                                  1708
AMAZON WEB SERVICES, INC.                                   1547
TESLA, INC.                                                  901
GOOGLE INC.                                                  545
FACEBOOK INC.                                                532
MICRON TECHNOLOGY, INC.                                      471
ADVANCED MICRO DEVICES, INC.                                 440
AMAZON.COM SERVICES, INC                                     381
NVIDIA CORPORATION                                           341
MICRONET IT SOLUTIONS, INC.                                  261
AMAZON SERVICES LLC                                          115
AMAZON ROBO

In [17]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Alphabet     10009
Microsoft     6040
Meta          2680
Apple         1708
Tesla          901
Micron         471
AMD            440
NVIDIA         341
Name: count, dtype: int64


In [19]:
df_clean = df.copy()

In [23]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["WORKSITE_CITY_1"] = (
    df["WORKSITE_CITY_1"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["WORKSITE_CITY_1"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
New York    1207
Seattle      793
Austin       520
San Jose      62
Dallas         6
Name: count, dtype: int64


In [27]:
print(df["WAGE_UNIT_OF_PAY_1"].value_counts())

WAGE_UNIT_OF_PAY_1
Year     2583
Hour        3
Month       2
Name: count, dtype: int64


In [35]:
import pandas as pd

# Convert wage values to numbers
df["WAGE_RATE_OF_PAY_FROM_1"] = pd.to_numeric(
    df["WAGE_RATE_OF_PAY_FROM_1"],
    errors="coerce"
)

# Standardize wage units
df["WAGE_UNIT_OF_PAY_1"] = (
    df["WAGE_UNIT_OF_PAY_1"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["WAGE_RATE_OF_PAY_FROM_1"]
    * df["WAGE_UNIT_OF_PAY_1"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "WAGE_RATE_OF_PAY_FROM_1",
    "WAGE_UNIT_OF_PAY_1",
    "Annual_Wage"
]].head(10))

       Company      City  WAGE_RATE_OF_PAY_FROM_1 WAGE_UNIT_OF_PAY_1  \
356       Meta  New York                172093.59               YEAR   
410   Alphabet  New York                191000.00               YEAR   
431       Meta  New York                139683.55               YEAR   
489      Apple    Austin                 90896.00               YEAR   
630       Meta  New York                129757.65               YEAR   
766       Meta  New York                175000.00               YEAR   
1095  Alphabet  New York                177000.00               YEAR   
1108  Alphabet  New York                230000.00               YEAR   
1128  Alphabet  New York                208896.00               YEAR   
1145  Alphabet  New York                191000.00               YEAR   

      Annual_Wage  
356     172093.59  
410     191000.00  
431     139683.55  
489      90896.00  
630     129757.65  
766     175000.00  
1095    177000.00  
1108    230000.00  
1128    208896.00  
1145   

In [37]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count    2.588000e+03
mean     1.433427e+05
std      5.729291e+04
min      5.759600e+04
25%      1.183377e+05
50%      1.400000e+05
75%      1.600000e+05
max      2.016000e+06
Name: Annual_Wage, dtype: float64


In [39]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [41]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

        City    Company  Number_of_Records  Median_Wage      Mean_Wage
0     Austin        AMD                172   105030.500  108648.215116
1     Austin   Alphabet                 82   115000.000  111360.207317
2     Austin      Apple                198   116979.000  116553.207071
3     Austin       Meta                 35   108000.000  117008.017714
4     Austin     Micron                  8   136468.410  143656.657500
5     Austin  Microsoft                  3   127204.000  141222.666667
6     Austin     NVIDIA                 22   122366.000  126870.990909
7     Dallas      Apple                  4   105113.000  103495.500000
8     Dallas       Meta                  2   135513.775  135513.775000
9   New York   Alphabet                920   142000.000  147563.421957
10  New York      Apple                  2   148558.500  148558.500000
11  New York       Meta                243   163000.000  166573.791193
12  New York  Microsoft                 40   152585.000  152818.975000
13  Ne

In [43]:
columns_to_keep = [
    "Company",
    "EMPLOYER_NAME",
    "JOB_TITLE",
    "SOC_TITLE",
    "City",
    "WAGE_RATE_OF_PAY_FROM_1",
    "WAGE_UNIT_OF_PAY_1",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2019.csv",
    index=False
)